<a href="https://colab.research.google.com/github/IrinAnd/Especializacion_DA_ITAcademy/blob/main/Sprint_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Carga y limpieza.



In [132]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

ruta= '/content/drive/MyDrive/sprint8_complex.xlsx'
df= pd.read_excel(ruta,header=3)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Homogenización de columnas, Eliminación de índice.

En este apartado se trabaja los dos primeros errores detectados: Nombre de columna erronéo y doble índice. Se ha reescrito el nombre de las columnas y eliminado el índice sobrante.

In [133]:
df.columns =[
    'index', 'nom', 'cognoms', 'dni', 'pais_origen', 'ciutat',
    'dia_naixement', 'mes_naixement', 'any_naixement', 'genere',
    'salari_mensual', 'fills', 'no_fills', 'grup_professional',
    'carrec', 'nombre_fills', 'te_cotxe', 'km_anuals',
    'consum_mitja', 'temperatura_mitjana'
            ]

df = df.drop(columns=['index'])
print(df)


          nom      cognoms        dni pais_origen       ciutat  dia_naixement  \
0       Joana  Gil Navarro  66722344X     Espanya   Valladolid           23.0   
1        Marc       MuÃ±oz  48840994W     Espanya      Alacant            8.0   
2         Noa        Serra  14308421X     Espanya      Alacant           27.0   
3         Pol          Gil  58586340F     Espanya      Sevilla           13.0   
4       David         Vila  82070937P     Espanya       Bilbao           29.0   
...       ...          ...        ...         ...          ...            ...   
1002     Alba       MuÃ±oz  97954019T     Espanya       Bilbao           25.0   
1003  Claudia   Gil Ferrer  55297384F     Espanya    ValÃ¨ncia            1.0   
1004    Oriol       Ferrer  52590001R     Espanya     Zaragoza           15.0   
1005      NaN          NaN        NaN         NaN          NaN            NaN   
1006      ???          ###       XXXX      Narnia  Desconeguda          -99.0   

      mes_naixement  any_na

# NIVEL 1 *EJERCICIO 1* : Exploración Inicial.

### Exploración inicial.

Durante la exploración inicial se han detectado  errores en las siguientes columnas: Nombre, apellido, país, salario mensual y cargo profesional a grandes rasgos se detectan distintos tipos de simbolos extraños.


In [134]:
print('1. Exploración inicial del dataset')
print('Filas y columnas:', df.shape[0],'Filas', df.shape[1],'Columnas')
print('Tipos de datos:', df.dtypes)
print('Duplicados:',df.duplicated().sum(), 'registros')
print('Valores nulos por columnas:',df.isna().all(axis=1).sum())

print('Muestra de datos:')
print(df.head(5))



1. Exploración inicial del dataset
Filas y columnas: 1007 Filas 19 Columnas
Tipos de datos: nom                     object
cognoms                 object
dni                     object
pais_origen             object
ciutat                  object
dia_naixement          float64
mes_naixement          float64
any_naixement          float64
genere                  object
salari_mensual          object
fills                   object
no_fills                object
grup_professional       object
carrec                  object
nombre_fills           float64
te_cotxe                object
km_anuals              float64
consum_mitja           float64
temperatura_mitjana    float64
dtype: object
Duplicados: 4 registros
Valores nulos por columnas: 1
Muestra de datos:
     nom      cognoms        dni pais_origen      ciutat  dia_naixement  \
0  Joana  Gil Navarro  66722344X     Espanya  Valladolid           23.0   
1   Marc       MuÃ±oz  48840994W     Espanya     Alacant            8.0   
2    Noa

##N1 *EJERCICIO* 2: Validaciones básicas y decisiones de eliminación.

Coche + gasolina anual

### Validació de símbolos extraños.

En este bloque se ha buscado resolver los problemas de Str detectados. Primero se ha decidido definir que texto se espera recibir, para señalar y devolver todos aquellos signos extraños que no deberían formar parte del texto. Una vez detectado los simbolos extraños creamos un diccionario con las palabras erroneas y sus debidas correciones.

In [135]:

import re

def tiene_simbolos_raros(texto):
    patron = r'[^a-zA-ZáéíóúñüÁÉÍÓÚÑÜ\s.]'
    return texto.astype(str).str.contains(patron, regex=True)

columnas = ['nom', 'cognoms', 'pais', 'grup_professional', 'ciutat']

for col in columnas:
    if col in df.columns:
        cantidad = tiene_simbolos_raros(df[col]).sum()
        print(f' Columna "{col}": {cantidad} registros con símbolos extraños')

        if cantidad > 0:
          filas_problematicas = df[tiene_simbolos_raros(df[col])]

          simbolos = set()
          for texto in filas_problematicas[col].astype(str):
              simbolos.update(re.findall(r'[^a-zA-ZáéíóúñüÁÉÍÓÚÑÜ\s.]', texto))

          print(f'Columna "{col}": {cantidad} registros con símbolos')
          print(f'Símbolos encontrados: {sorted(simbolos)}')
          print(f'textos: {filas_problematicas[col].head().tolist()}\n')


 Columna "nom": 129 registros con símbolos extraños
Columna "nom": 129 registros con símbolos
Símbolos encontrados: ['?', '©', '\xad', 'º', 'Ã', '‰']
textos: ['AdriÃ\xa0', 'AdriÃ\xa0', 'ChloÃ©', 'AdriÃ\xa0', 'AdriÃ\xa0']

 Columna "cognoms": 377 registros con símbolos extraños
Columna "cognoms": 377 registros con símbolos
Símbolos encontrados: ['#', '\x81', '¡', '¤', '¨', '©', '\xad', '±', '³', '¶', '¼', 'Ã']
textos: ['MuÃ±oz', 'GÃ³mez Vila', 'LÃ³pez', 'GarcÃ\xada', 'Torres DÃ\xadaz']

 Columna "grup_professional": 0 registros con símbolos extraños
 Columna "ciutat": 160 registros con símbolos extraños
Columna "ciutat": 160 registros con símbolos
Símbolos encontrados: ['¨', '±', '¶', '¸', '¼', 'Ã']
textos: ['ValÃ¨ncia', 'MÃ\xa0laga', 'A CoruÃ±a', 'MÃ\xa0laga', 'A CoruÃ±a']



In [136]:
correcciones = {
    'Ã±': 'ñ',
     'Ã':'a',
    'Ã¡': 'á',
    'Ã©': 'é',
    'Ã­': 'í',
    'Ã³': 'ó',
    'Ãº': 'ú',
    'Ã¼': 'ü',
    'Ã¨': 'e' ,
    'Ã¨': 'é' ,
    'a¨':'e',
    'Ã§':'ç',
    '§':'ç',
    'Ã¶':'öl',
    '¶l': 'öl',
    '©':'',
    '%': '',
    '¡':'',
    '  ': ' '
}

def corregir_nombres(texto):
    if pd.isna(texto):
        return texto
    texto = str(texto)
    for errado, correcto in correcciones.items():
        texto = texto.replace(errado, correcto)
    return texto


columnas_a_corregir = ['nom', 'cognoms', 'pais_origen', 'grup_professional', 'ciutat', 'carrec']

for col in columnas_a_corregir:
    if col in df.columns:
        df[col] = df[col].apply(corregir_nombres)
        print(' Columna ',col, 'corregida')

display(df)

 Columna  nom corregida
 Columna  cognoms corregida
 Columna  pais_origen corregida
 Columna  grup_professional corregida
 Columna  ciutat corregida
 Columna  carrec corregida


,nom,cognoms,dni,pais_origen,ciutat,dia_naixement,mes_naixement,any_naixement,genere,salari_mensual,fills,no_fills,grup_professional,carrec,nombre_fills,te_cotxe,km_anuals,consum_mitja,temperatura_mitjana
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 â‚¬",True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1
1,Marc,Muñoz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 â‚¬",NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7
2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,D,1358 euros,True,NaN,Grup A,Tecnic IT,4.0,NaN,NaN,NaN,16.7
3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,H,"1,478 â‚¬",True,NaN,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3
4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,"1,284 â‚¬",NaN,True,Grup B,Administratiu,NaN,True,NaN,11.8,13.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,Alba,Muñoz,97954019T,Espanya,Bilbao,25.0,10.0,1983.0,D,"1,409 â‚¬",NaN,True,Grup B,Responsable de vendes,0.0,NaN,NaN,NaN,14.7
1003,Claudia,Gil Ferrer,55297384F,Espanya,Valencia,1.0,11.0,1981.0,H,1.789 â‚¬,NaN,True,Grup B,Responsable de vendes,0.0,True,29952.0,9.2,20.3
1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786â‚¬,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9
1005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Validación DNI válido.
En este bloque se validó la columna dni según el formato oficial de españa.  se confirmó que no existen DNIs inválidos, 0 registros erróneos.
La siguiente comprobación fueron lod duplicados. Se detectaron 15 registros  y se eliminaron, manteniendo únicamente el primer registro de cada DNI. Se ha seguido ese criterio ya que el DNI es un documento único y se interpreta que la primera inscripción es la intencionada.

In [137]:

def dni_validado(dni):
    if pd.isna(dni):
        return False
    dni = str(dni).strip()
    if len(dni) != 9:
        return False
    if not dni[:8].isdigit():
        return False
    if not dni[8].isupper():
        return False
    return True


df['dni_valido'] = df['dni'].apply(dni_validado)
dni_erroneos_df = df[~df['dni_valido']]

print(' DNIs inválidos encontrados:', len(dni_erroneos_df))

if len(dni_erroneos_df) > 0:
    print('Registros con DNI inválido:')
    display(dni_erroneos_df[['dni', 'nom', 'cognoms']].head(10))
else:
    print('No se encontraron DNIs inválidos.')


 DNIs inválidos encontrados: 2
Registros con DNI inválido:


,dni,nom,cognoms
1005,NaN,NaN,NaN
1006,XXXX,???,###


In [138]:

dnis_duplicados = df['dni'].duplicated().sum()

print('Número de DNIs duplicados:', dnis_duplicados)

if dnis_duplicados > 0:
    duplicados = df[df['dni'].duplicated(keep=False)]
    display(duplicados.sort_values(by='dni')[['dni', 'nom', 'cognoms']])

    df = df.drop_duplicates(subset=['dni'], keep='first')
    print('Se mantuvieron solo los primeros registros con DNI duplicado.')
    print('Registros restantes después de eliminar duplicados:', len(df))
else:
    print('No se encontraron DNIs duplicados.')

Número de DNIs duplicados: 15


,dni,nom,cognoms
604,15909601K,Laia,Navarro
250,15909601K,Laia,Navarro Vila
759,16618423V,Oriol,Ga³mez
758,16618423V,Oriol,Ga³mez
754,16618423V,Oriol,Ga³mez
654,16618423V,Oriol,Ga³mez
692,49529438G,Alexia,Parez Da­az
558,49529438G,Alexia,Parez Da­az
757,52590001R,Oriol,Ferrer
1004,52590001R,Oriol,Ferrer


Se mantuvieron solo los primeros registros con DNI duplicado.
Registros restantes después de eliminar duplicados: 992


###Validación de duplicados| Eliminación fila vacía | Eliminación fila incoherente.
Se eliminaron las filas 1005 y 1006 previamente detectadas. La primera presentaba valores nulos en todas las columnas y la segunda contenía información incoherente. Ambas fueron descartadas por no aportar valor al análisis y por ser irrecuperables sin fabricar datos.

In [139]:
duplicados = df.duplicated().sum()
df=df.drop_duplicates()
print('Se eliminaron', duplicados, 'registros duplicados')


Se eliminaron 0 registros duplicados


In [140]:
df = df.drop(index=[1005, 1006])
print('Se eliminaron', duplicados, 'registros duplicados')

display(df)


Se eliminaron 0 registros duplicados


,nom,cognoms,dni,pais_origen,ciutat,dia_naixement,mes_naixement,any_naixement,genere,salari_mensual,fills,no_fills,grup_professional,carrec,nombre_fills,te_cotxe,km_anuals,consum_mitja,temperatura_mitjana,dni_valido
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 â‚¬",True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1,True
1,Marc,Muñoz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 â‚¬",NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7,True
2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,D,1358 euros,True,NaN,Grup A,Tecnic IT,4.0,NaN,NaN,NaN,16.7,True
3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,H,"1,478 â‚¬",True,NaN,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3,True
4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,"1,284 â‚¬",NaN,True,Grup B,Administratiu,NaN,True,NaN,11.8,13.1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,Luca­a,Roca Vila,21702378D,Espanya,Alacant,32.0,0.0,1940.0,NaN,1158â‚¬,NaN,True,Grup A,Data Analyst,NaN,NaN,NaN,NaN,19.4,True
997,Jordi,Hernandez,67755039Y,Espanya,Zaragoza,12.0,10.0,1961.0,Dona,"1,074 â‚¬",NaN,NaN,Grup A,Data Analyst,1.0,True,80000.0,12.0,12.0,True
998,Chloa,Dubois,66354268T,Franaça,Nice,2.0,12.0,1958.0,H,"1,954 â‚¬",True,NaN,Grup B,Data Analyst,1.0,NaN,NaN,NaN,11.7,True
999,Adria,Vila Da­az,57511543T,Espanya,Palma,31.0,12.0,1991.0,H,"2,010 â‚¬",True,NaN,Grup C,Analista,1.0,False,NaN,NaN,-10.0,True


### Fechas:Validacion de fechas:

En este bloque se validaron las fechas de nacimiento, detectando valores imposibles en día, mes y año. Se eligió el año 2009 como límite porque representa la edad mínima legal para trabajar en España (16 años en 2025). Los registros incoherentes fueron imputados con la falsa fecha 01/01/2009, para evitar eliminar filas y conservar el resto de información. Para acabar, se creó la columna data_naixement para unificar la información en una columna.

In [141]:
def limpiar_dia_mes(df):
    df = df.copy()

    df['dia_naixement'] = pd.to_numeric(df['dia_naixement'], errors='coerce').astype('Int64')
    df['mes_naixement'] = pd.to_numeric(df['mes_naixement'], errors='coerce').astype('Int64')


    condicion_imposible = (
        df['dia_naixement'].isna() |
        ~df['dia_naixement'].between(1, 31) |
        df['mes_naixement'].isna() |
        ~df['mes_naixement'].between(1, 12)
    )


    df.loc[condicion_imposible, 'dia_naixement'] = 1
    df.loc[condicion_imposible, 'mes_naixement'] = 1
    df.loc[condicion_imposible, 'any_naixement'] = 2009

    print(' Se imputaron', condicion_imposible.sum(), 'registros con día/mes imposible con fecha 01/01/2009')

    return df


In [142]:
def limpiar_any_naixement(df):

    df = df.copy()
    año_minimo = 2009

    condicion_imposible = df['any_naixement'] > año_minimo

    if condicion_imposible.sum() > 0:
        df.loc[condicion_imposible, 'any_naixement'] = 2009
        print(' Se imputaron', condicion_imposible.sum(), 'años de nacimiento imposibles con el valor 2009')

    return df

In [143]:
def crear_fecha_nacimiento(df):
    df = df.copy()

    df['any_naixement'] = pd.to_numeric(df['any_naixement'], errors='coerce').astype('Int64')

    df['data_naixement'] = pd.to_datetime(
        df[['any_naixement', 'mes_naixement', 'dia_naixement']].rename(
            columns={'any_naixement': 'year', 'mes_naixement': 'month', 'dia_naixement': 'day'}
        ),
        errors='coerce'
    )

    df = df.drop(columns=['dia_naixement', 'mes_naixement', 'any_naixement'], errors='ignore')

    print('Columna "data_naixement" creada correctamente. Valores nulos:', df['data_naixement'].isnull().sum())
    return df

In [144]:

df = limpiar_dia_mes(df)
df = limpiar_any_naixement(df)
df = crear_fecha_nacimiento(df)


display(df)

 Se imputaron 77 registros con día/mes imposible con fecha 01/01/2009
Columna "data_naixement" creada correctamente. Valores nulos: 0


,nom,cognoms,dni,pais_origen,ciutat,genere,salari_mensual,fills,no_fills,grup_professional,carrec,nombre_fills,te_cotxe,km_anuals,consum_mitja,temperatura_mitjana,dni_valido,data_naixement
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,"1,469 â‚¬",True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1,True,1958-03-23
1,Marc,Muñoz,48840994W,Espanya,Alacant,H,"2,718 â‚¬",NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7,True,1960-11-08
2,Noa,Serra,14308421X,Espanya,Alacant,D,1358 euros,True,NaN,Grup A,Tecnic IT,4.0,NaN,NaN,NaN,16.7,True,1961-04-27
3,Pol,Gil,58586340F,Espanya,Sevilla,H,"1,478 â‚¬",True,NaN,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3,True,1985-10-13
4,David,Vila,82070937P,Espanya,Bilbao,Dona,"1,284 â‚¬",NaN,True,Grup B,Administratiu,NaN,True,NaN,11.8,13.1,True,1965-11-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,Luca­a,Roca Vila,21702378D,Espanya,Alacant,NaN,1158â‚¬,NaN,True,Grup A,Data Analyst,NaN,NaN,NaN,NaN,19.4,True,2009-01-01
997,Jordi,Hernandez,67755039Y,Espanya,Zaragoza,Dona,"1,074 â‚¬",NaN,NaN,Grup A,Data Analyst,1.0,True,80000.0,12.0,12.0,True,1961-10-12
998,Chloa,Dubois,66354268T,Franaça,Nice,H,"1,954 â‚¬",True,NaN,Grup B,Data Analyst,1.0,NaN,NaN,NaN,11.7,True,1958-12-02
999,Adria,Vila Da­az,57511543T,Espanya,Palma,H,"2,010 â‚¬",True,NaN,Grup C,Analista,1.0,False,NaN,NaN,-10.0,True,1991-12-31


###Normalización del genero.

En este apartado se detectó una gran disparidad en los valores de la columna genero, con múltiples formas de referirse al mismo concepto . Para solucionar esta inconsistencia, se creó un diccionario de mapeo que unifica todas las variantes en dos categorías representativas: H para hombre y M para mujer, y  "No especificado" en los casos nulos o sin respuesta.Se ha decidido seguir esta lógica porque no hay suficientes criterios para decidir el genero de los empleados fuera de la columna 'nombre'.

In [145]:

def normalizar_genere(df):
    df = df.copy()


    df['genere'] = df['genere'].astype(str).str.strip().str.lower()


    mapeo_genere = {
       'h': 'Home',
        'home': 'Home',
        'm': 'Home',
        'd': 'Dona',
        'dona': 'Dona',
        'f': 'Dona',
        'a': 'No especificado',
        'nc': 'No especificado',
        'nan': 'No especificado',
        '': 'No especificado',
        'none': 'No especificado'

    }


    df['genere'] = df['genere'].map(mapeo_genere)
    df['genere'] = df['genere'].fillna('No especificado')
    print(df['genere'].value_counts())
    return df


In [146]:
df= normalizar_genere(df)
display(df)

genere
Dona               545
Home               359
No especificado     86
Name: count, dtype: int64


,nom,cognoms,dni,pais_origen,ciutat,genere,salari_mensual,fills,no_fills,grup_professional,carrec,nombre_fills,te_cotxe,km_anuals,consum_mitja,temperatura_mitjana,dni_valido,data_naixement
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,"1,469 â‚¬",True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1,True,1958-03-23
1,Marc,Muñoz,48840994W,Espanya,Alacant,Home,"2,718 â‚¬",NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7,True,1960-11-08
2,Noa,Serra,14308421X,Espanya,Alacant,Dona,1358 euros,True,NaN,Grup A,Tecnic IT,4.0,NaN,NaN,NaN,16.7,True,1961-04-27
3,Pol,Gil,58586340F,Espanya,Sevilla,Home,"1,478 â‚¬",True,NaN,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3,True,1985-10-13
4,David,Vila,82070937P,Espanya,Bilbao,Dona,"1,284 â‚¬",NaN,True,Grup B,Administratiu,NaN,True,NaN,11.8,13.1,True,1965-11-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,Luca­a,Roca Vila,21702378D,Espanya,Alacant,No especificado,1158â‚¬,NaN,True,Grup A,Data Analyst,NaN,NaN,NaN,NaN,19.4,True,2009-01-01
997,Jordi,Hernandez,67755039Y,Espanya,Zaragoza,Dona,"1,074 â‚¬",NaN,NaN,Grup A,Data Analyst,1.0,True,80000.0,12.0,12.0,True,1961-10-12
998,Chloa,Dubois,66354268T,Franaça,Nice,Home,"1,954 â‚¬",True,NaN,Grup B,Data Analyst,1.0,NaN,NaN,NaN,11.7,True,1958-12-02
999,Adria,Vila Da­az,57511543T,Espanya,Palma,Home,"2,010 â‚¬",True,NaN,Grup C,Analista,1.0,False,NaN,NaN,-10.0,True,1991-12-31


# N1 *EJERCICIO* 3: Transformaciones necesarias y preparación del dataset.

### Validación del salario.
En este bloque se hizo la limpieza de la columna salari_mensual, que contenía valores en diferentes formatos . Se creó una función que combina un diccionario de mapeo y expresiones regulares para convertir todos los valores a formato numérico.
Se imputaron los valores nulos de la columna salari_mensual utilizando la mediana del salario por grup_professional. Este criterio se eligió porque el salario está relacionado con el grupo profesional al que pertenece el trabajador. De esta manera se conserva toda la información útil de los registros sin inventar datos arbitrarios.

In [147]:

import re

def convertir_salario(texto):
    if pd.isna(texto):
        return None

    texto = str(texto).strip().lower()


    mapeo = {
        'mil vuit-cents': 1800,
        'dos mil': 2000,
        'tres mil': 3000,
        'mil': 1000,
        'n/d': None,
        'nan': None,
        '': None
    }

    if texto in mapeo:
        return mapeo[texto]


    texto_limpio = re.sub(r'[^0-9.,]', '', texto)
    texto_limpio = texto_limpio.replace(',', '.')

    try:
        return float(texto_limpio)
    except:
        return None



df['salari_mensual'] = df['salari_mensual'].apply(convertir_salario)

print("Distribución después de limpieza de salario:")
print(df['salari_mensual'].value_counts(dropna=False))
print(f"\nTotal de NaN restantes: {df['salari_mensual'].isnull().sum()}")
display(df)

Distribución después de limpieza de salario:
salari_mensual
NaN         14
1800.000     9
2000.000     7
3000.000     6
1.469        4
            ..
1507.000     1
1916.000     1
1265.000     1
1.807        1
1.092        1
Name: count, Length: 802, dtype: int64

Total de NaN restantes: 14


,nom,cognoms,dni,pais_origen,ciutat,genere,salari_mensual,fills,no_fills,grup_professional,carrec,nombre_fills,te_cotxe,km_anuals,consum_mitja,temperatura_mitjana,dni_valido,data_naixement
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,1.469,True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1,True,1958-03-23
1,Marc,Muñoz,48840994W,Espanya,Alacant,Home,2.718,NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7,True,1960-11-08
2,Noa,Serra,14308421X,Espanya,Alacant,Dona,1358.000,True,NaN,Grup A,Tecnic IT,4.0,NaN,NaN,NaN,16.7,True,1961-04-27
3,Pol,Gil,58586340F,Espanya,Sevilla,Home,1.478,True,NaN,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3,True,1985-10-13
4,David,Vila,82070937P,Espanya,Bilbao,Dona,1.284,NaN,True,Grup B,Administratiu,NaN,True,NaN,11.8,13.1,True,1965-11-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,Luca­a,Roca Vila,21702378D,Espanya,Alacant,No especificado,1158.000,NaN,True,Grup A,Data Analyst,NaN,NaN,NaN,NaN,19.4,True,2009-01-01
997,Jordi,Hernandez,67755039Y,Espanya,Zaragoza,Dona,1.074,NaN,NaN,Grup A,Data Analyst,1.0,True,80000.0,12.0,12.0,True,1961-10-12
998,Chloa,Dubois,66354268T,Franaça,Nice,Home,1.954,True,NaN,Grup B,Data Analyst,1.0,NaN,NaN,NaN,11.7,True,1958-12-02
999,Adria,Vila Da­az,57511543T,Espanya,Palma,Home,2.010,True,NaN,Grup C,Analista,1.0,False,NaN,NaN,-10.0,True,1991-12-31


In [148]:
# x grupo salarial
df['salari_mensual'] = df.groupby('grup_professional')['salari_mensual'].transform(
    lambda x: x.fillna(x.median())
)

print('NaN en salari_mensual depúes de imputar:',df['salari_mensual'].isnull().sum())
print('Distribución final de salari_mensual:')
print(df['salari_mensual'].describe())
display(df)

NaN en salari_mensual depúes de imputar: 0
Distribución final de salari_mensual:
count     990.000000
mean      619.881432
std       820.631951
min         1.001000
25%         1.469000
50%         2.522000
75%      1061.000000
max      3672.000000
Name: salari_mensual, dtype: float64


,nom,cognoms,dni,pais_origen,ciutat,genere,salari_mensual,fills,no_fills,grup_professional,carrec,nombre_fills,te_cotxe,km_anuals,consum_mitja,temperatura_mitjana,dni_valido,data_naixement
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,1.469,True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1,True,1958-03-23
1,Marc,Muñoz,48840994W,Espanya,Alacant,Home,2.718,NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7,True,1960-11-08
2,Noa,Serra,14308421X,Espanya,Alacant,Dona,1358.000,True,NaN,Grup A,Tecnic IT,4.0,NaN,NaN,NaN,16.7,True,1961-04-27
3,Pol,Gil,58586340F,Espanya,Sevilla,Home,1.478,True,NaN,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3,True,1985-10-13
4,David,Vila,82070937P,Espanya,Bilbao,Dona,1.284,NaN,True,Grup B,Administratiu,NaN,True,NaN,11.8,13.1,True,1965-11-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,Luca­a,Roca Vila,21702378D,Espanya,Alacant,No especificado,1158.000,NaN,True,Grup A,Data Analyst,NaN,NaN,NaN,NaN,19.4,True,2009-01-01
997,Jordi,Hernandez,67755039Y,Espanya,Zaragoza,Dona,1.074,NaN,NaN,Grup A,Data Analyst,1.0,True,80000.0,12.0,12.0,True,1961-10-12
998,Chloa,Dubois,66354268T,Franaça,Nice,Home,1.954,True,NaN,Grup B,Data Analyst,1.0,NaN,NaN,NaN,11.7,True,1958-12-02
999,Adria,Vila Da­az,57511543T,Espanya,Palma,Home,2.010,True,NaN,Grup C,Analista,1.0,False,NaN,NaN,-10.0,True,1991-12-31


###Hijos: Validación de las celdas.
Existían tres columnas relacionadas con la descendencia: fills, no_fills y nombre_fills. Se decidió unificar toda la información en una única columna nombre_fills para evitar redundancia.
Se normalizaron los valores y se imputaron como 1 aquellos casos donde constaba que tenían hijos: pero no se indicaba la cantidad. Los valores nulos restantes se imputaron como 0.

In [149]:

df['nombre_fills'] = pd.to_numeric(df['nombre_fills'], errors='coerce')

df['nombre_fills'] = df['nombre_fills'].round(0).astype('Int64')

condicion = (df['fills'] == True) & (df['nombre_fills'].isin([0, pd.NA, None]))
df.loc[condicion, 'nombre_fills'] = 1

df['nombre_fills'] = df['nombre_fills'].fillna(0)


df = df.drop(columns=['fills', 'no_fills'], errors='ignore')

print("Distribución final de 'nombre_fills':")
print(df['nombre_fills'].value_counts().sort_index())
print('Total de registros:', len(df))
display(df)



Distribución final de 'nombre_fills':
nombre_fills
0    562
1    206
2     79
3     64
4     79
Name: count, dtype: Int64
Total de registros: 990


,nom,cognoms,dni,pais_origen,ciutat,genere,salari_mensual,grup_professional,carrec,nombre_fills,te_cotxe,km_anuals,consum_mitja,temperatura_mitjana,dni_valido,data_naixement
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,1.469,Grup B,Cap de projecte,3,True,32108.0,25.0,10.1,True,1958-03-23
1,Marc,Muñoz,48840994W,Espanya,Alacant,Home,2.718,Grup C,Senior analyst,0,True,19496.0,10.4,18.7,True,1960-11-08
2,Noa,Serra,14308421X,Espanya,Alacant,Dona,1358.000,Grup A,Tecnic IT,4,NaN,NaN,NaN,16.7,True,1961-04-27
3,Pol,Gil,58586340F,Espanya,Sevilla,Home,1.478,Grup B,Data Analyst,2,False,NaN,NaN,18.3,True,1985-10-13
4,David,Vila,82070937P,Espanya,Bilbao,Dona,1.284,Grup B,Administratiu,0,True,NaN,11.8,13.1,True,1965-11-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,Luca­a,Roca Vila,21702378D,Espanya,Alacant,No especificado,1158.000,Grup A,Data Analyst,0,NaN,NaN,NaN,19.4,True,2009-01-01
997,Jordi,Hernandez,67755039Y,Espanya,Zaragoza,Dona,1.074,Grup A,Data Analyst,1,True,80000.0,12.0,12.0,True,1961-10-12
998,Chloa,Dubois,66354268T,Franaça,Nice,Home,1.954,Grup B,Data Analyst,1,NaN,NaN,NaN,11.7,True,1958-12-02
999,Adria,Vila Da­az,57511543T,Espanya,Palma,Home,2.010,Grup C,Analista,1,False,NaN,NaN,-10.0,True,1991-12-31


### Normalización de Cargo (Carrec)

En este apartado se observó una gran variedad títulos para cargos similares. Por ello, se ha obtado por crear un diccionario de mapeo donde se estandarizan los cargos.

In [150]:
def normalizar_carrec(df):
    df = df.copy()


    df['carrec'] = df['carrec'].astype(str).str.strip().str.title()

    mapeo_cargo = {
        # Grupo Administrativo
        'Administratiu': 'Administratiu',
        'Admin.': 'Administratiu',

        # Grupo Analista
        'Analista': 'Analista',
        'Analista Junior': 'Analista',
        'Analyst': 'Analista',
        'Data Analyst': 'Analista',
        'Senior Analyst': 'Analista',

        # Grupo Técnico IT
        'Tècnic It': 'Tècnic IT',
        'Tecnic It': 'Tècnic IT',
        'Tècnic IT': 'Tècnic IT',

        # Grupo Project Manager
        'Cap De Projecte': 'Cap de Projecte',
        'Project Lead': 'Cap de Projecte',

        # Responsable de ventas
        'Responsable De Vendes': 'Responsable de Vendes'
    }


    df['carrec'] = df['carrec'].map(mapeo_cargo).fillna(df['carrec'])

    print("Distribución después de normalizar 'carrec':")
    print(df['carrec'].value_counts())

    return df



df = normalizar_carrec(df)
display(df)

Distribución después de normalizar 'carrec':
carrec
Analista                 508
Tècnic IT                128
Cap de Projecte          124
Responsable de Vendes    121
Administratiu            109
Name: count, dtype: int64


,nom,cognoms,dni,pais_origen,ciutat,genere,salari_mensual,grup_professional,carrec,nombre_fills,te_cotxe,km_anuals,consum_mitja,temperatura_mitjana,dni_valido,data_naixement
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,1.469,Grup B,Cap de Projecte,3,True,32108.0,25.0,10.1,True,1958-03-23
1,Marc,Muñoz,48840994W,Espanya,Alacant,Home,2.718,Grup C,Analista,0,True,19496.0,10.4,18.7,True,1960-11-08
2,Noa,Serra,14308421X,Espanya,Alacant,Dona,1358.000,Grup A,Tècnic IT,4,NaN,NaN,NaN,16.7,True,1961-04-27
3,Pol,Gil,58586340F,Espanya,Sevilla,Home,1.478,Grup B,Analista,2,False,NaN,NaN,18.3,True,1985-10-13
4,David,Vila,82070937P,Espanya,Bilbao,Dona,1.284,Grup B,Administratiu,0,True,NaN,11.8,13.1,True,1965-11-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,Luca­a,Roca Vila,21702378D,Espanya,Alacant,No especificado,1158.000,Grup A,Analista,0,NaN,NaN,NaN,19.4,True,2009-01-01
997,Jordi,Hernandez,67755039Y,Espanya,Zaragoza,Dona,1.074,Grup A,Analista,1,True,80000.0,12.0,12.0,True,1961-10-12
998,Chloa,Dubois,66354268T,Franaça,Nice,Home,1.954,Grup B,Analista,1,NaN,NaN,NaN,11.7,True,1958-12-02
999,Adria,Vila Da­az,57511543T,Espanya,Palma,Home,2.010,Grup C,Analista,1,False,NaN,NaN,-10.0,True,1991-12-31


### Kilometraje anual
En este apartado se ha identificado  15 registros con coche pero sin kilometraje anual  consum mitjà. En lugar de eliminarlos, se ha optado por imputar los valores ausentes utilizando la mediana por grup_professional. Es razonable pensar que un mismo grupo profesional comparta kilometraje y consumo.
El criterio ha seguir ha sido el siguiente:
Cuando no tiene coche imputamos un valor 0
Cuando tiene coche pero falta un valor se le imputa la media por grupo profesional.

In [151]:

df['te_cotxe'] = df['te_cotxe'].fillna(False)

print('NaN en Km antes:', df['km_anuals'].isnull().sum() if 'km_anuals' in df.columns else "Columna no existe")
print('NaN en Consum antes:', df['consum_mitja'].isnull().sum() if 'consum_mitja' in df.columns else "Columna no existe")

# Si NO tiene coche = 0
df.loc[df['te_cotxe'] == False, ['km_anuals', 'consum_mitja']] = 0

# Contrario: imputación por grupo profesional
df['km_anuals'] = df.groupby('grup_professional')['km_anuals'].transform(
    lambda x: x.fillna(x.median())
)

df['consum_mitja'] = df.groupby('grup_professional')['consum_mitja'].transform(
    lambda x: x.fillna(x.median())
)


df['km_anuals'] = df['km_anuals'].fillna(df['km_anuals'].median())
df['consum_mitja'] = df['consum_mitja'].fillna(df['consum_mitja'].median())

print('NaN después de imputar:')
print('km_anuals:', df['km_anuals'].isnull().sum())
print('consum_mitja:', df['consum_mitja'].isnull().sum())

NaN en Km antes: 680
NaN en Consum antes: 733
NaN después de imputar:
km_anuals: 0
consum_mitja: 0


/tmp/ipykernel_3924/3191163930.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['te_cotxe'] = df['te_cotxe'].fillna(False)


In [152]:
display(df)

,nom,cognoms,dni,pais_origen,ciutat,genere,salari_mensual,grup_professional,carrec,nombre_fills,te_cotxe,km_anuals,consum_mitja,temperatura_mitjana,dni_valido,data_naixement
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,1.469,Grup B,Cap de Projecte,3,True,32108.0,25.0,10.1,True,1958-03-23
1,Marc,Muñoz,48840994W,Espanya,Alacant,Home,2.718,Grup C,Analista,0,True,19496.0,10.4,18.7,True,1960-11-08
2,Noa,Serra,14308421X,Espanya,Alacant,Dona,1358.000,Grup A,Tècnic IT,4,False,0.0,0.0,16.7,True,1961-04-27
3,Pol,Gil,58586340F,Espanya,Sevilla,Home,1.478,Grup B,Analista,2,False,0.0,0.0,18.3,True,1985-10-13
4,David,Vila,82070937P,Espanya,Bilbao,Dona,1.284,Grup B,Administratiu,0,True,0.0,11.8,13.1,True,1965-11-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,Luca­a,Roca Vila,21702378D,Espanya,Alacant,No especificado,1158.000,Grup A,Analista,0,False,0.0,0.0,19.4,True,2009-01-01
997,Jordi,Hernandez,67755039Y,Espanya,Zaragoza,Dona,1.074,Grup A,Analista,1,True,80000.0,12.0,12.0,True,1961-10-12
998,Chloa,Dubois,66354268T,Franaça,Nice,Home,1.954,Grup B,Analista,1,False,0.0,0.0,11.7,True,1958-12-02
999,Adria,Vila Da­az,57511543T,Espanya,Palma,Home,2.010,Grup C,Analista,1,False,0.0,0.0,-10.0,True,1991-12-31


### Temperatura agrupada por ciudad.

En este bloque se validó la columna temperatura_mitjana. Dado que esta variable depende principalmente de la ubicación geográfica, se decidió imputar los valores faltantes utilizando la media por ciudad.



In [153]:

df['temperatura_mitjana'] = df.groupby('ciutat')['temperatura_mitjana'].transform('mean')

df['temperatura_mitjana'] = df['temperatura_mitjana'].round(2)

display(df.head())

,nom,cognoms,dni,pais_origen,ciutat,genere,salari_mensual,grup_professional,carrec,nombre_fills,te_cotxe,km_anuals,consum_mitja,temperatura_mitjana,dni_valido,data_naixement
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,1.469,Grup B,Cap de Projecte,3,True,32108.0,25.0,13.41,True,1958-03-23
1,Marc,Muñoz,48840994W,Espanya,Alacant,Home,2.718,Grup C,Analista,0,True,19496.0,10.4,17.38,True,1960-11-08
2,Noa,Serra,14308421X,Espanya,Alacant,Dona,1358.000,Grup A,Tècnic IT,4,False,0.0,0.0,17.38,True,1961-04-27
3,Pol,Gil,58586340F,Espanya,Sevilla,Home,1.478,Grup B,Analista,2,False,0.0,0.0,18.37,True,1985-10-13
4,David,Vila,82070937P,Espanya,Bilbao,Dona,1.284,Grup B,Administratiu,0,True,0.0,11.8,13.37,True,1965-11-29


## N1 EJERERCICIO 4:

En este bloque se construyeron tablas resumen para obtener una primera visión global del dataset.

En la primera tabla  se agrupa los salarios por género y se muestra las diferencias de retribución media entre los grupos.
En la segunda, la tabla pivote, se agrupa el salario por género y país, para entender mejor esta intersección. Finalmentes, como los tamaños de los grupos varían, se crea una tercera tabla donde se calcula el salario relativo respecto a la media general.

In [154]:
resumen_salario_genero = df.groupby('genere')['salari_mensual'].agg(
    Media='mean',
    Mediana='median',
    Mínimo='min',
    Máximo='max',
    Cantidad='count'
).round(2)


resumen_salario_genero = resumen_salario_genero.sort_values(by='Media', ascending=False)

print(' Resumen salarial por Género (group by media):')
display(resumen_salario_genero)


 Resumen salarial por Género (group by media):


,Media,Mediana,Mínimo,Máximo,Cantidad
genere,,,,,
Home,658.51,2.68,1.00,3672.0,359
Dona,599.08,2.34,1.00,3369.0,545
No especificado,590.47,2.65,1.02,3600.0,86


In [155]:

pivote = pd.pivot_table(
    df,
    values='salari_mensual',
    index='genere',
    columns='pais_origen',
    aggfunc='mean',
    margins=True,
    margins_name='Total'
).round(2)

print('Salario medio por Género y País de origen:')
display(pivote)

Salario medio por Género y País de origen:


pais_origen,Alemanya,Espanya,Franaça,Ita lia,Noruega,Total
genere,,,,,,
Dona,443.78,579.11,875.23,714.97,299.11,599.08
Home,941.71,679.04,589.96,523.90,581.29,658.51
No especificado,451.20,594.38,734.66,567.75,1.24,590.47
Total,558.50,617.59,759.32,634.51,410.30,619.88


In [156]:

salario_global = df['salari_mensual'].mean()

df['salario_relativo'] = (df['salari_mensual'] / salario_global * 100).round(1)

print('Salario relativo al promedio general (%):')
relativo = df.groupby('genere')['salario_relativo'].mean().round(1).sort_values(ascending=False)
display(relativo)

Salario relativo al promedio general (%):


,salario_relativo
genere,
Home,106.2
Dona,96.6
No especificado,95.3
